<a href="https://colab.research.google.com/github/LCaravaggio/FelicidadDesigualdad/blob/main/Sensibilidad.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Imágenes

In [1]:
from google.colab import userdata
import json

!mkdir ~/.kaggle
!touch ~/.kaggle/kaggle.json

api_token = {
    'username': userdata.get('KAGGLE_USER'),
    'key': userdata.get('KAGGLE_KEY')}
with open('/root/.kaggle/kaggle.json', 'w') as file:
    json.dump(api_token, file)

!chmod 600 ~/.kaggle/kaggle.json


import kagglehub
eph = kagglehub.dataset_download("leonardocaravaggio/ge-images5")
ocde = kagglehub.dataset_download("leonardocaravaggio/ge-images4")
ocde2 = kagglehub.dataset_download("leonardocaravaggio/ge-images6")

100%|██████████| 0.99G/0.99G [00:20<00:00, 51.0MB/s]

Extracting files...


100%|██████████| 3.21G/3.21G [00:42<00:00, 80.4MB/s]

Extracting files...


100%|██████████| 1.57G/1.57G [00:20<00:00, 80.7MB/s]

Extracting files...


# Modelo

In [189]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import numpy as np
import torch.nn.functional as F
import os
import timm
import torch.nn as nn

# Cargar MobileNetV2 hasta la capa 10
model = models.mobilenet_v2(pretrained=True)
model = nn.Sequential(*list(model.features[:7]))

#MobileNet V3
#model =models.mobilenet_v3_large(pretrained=True)
#model = nn.Sequential(*list(model.features[:10]))

#ResNet18
#resnet18 = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
#model = nn.Sequential(*list(resnet18.children())[:6])

#EfficienNet
#efficientnet = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
#model = nn.Sequential(*list(efficientnet.features[:6]))

#SwinFormer
#swin = models.swin_t(weights=models.Swin_T_Weights.IMAGENET1K_V1)
#model = nn.Sequential(*list(swin.features[:8]))

model.eval()

# Transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

def _global_pool(feats: torch.Tensor) -> torch.Tensor:
    # Acepta [B,C,H,W] o [B,L,C]
    if feats.dim() == 4:
        # mapa -> avg pool espacial -> [B, C]
        return F.adaptive_avg_pool2d(feats, (1, 1)).squeeze(-1).squeeze(-1)
    elif feats.dim() == 3:
        # tokens -> promedio sobre L -> [B, C]
        return feats.mean(dim=1)
    else:
        raise RuntimeError(f"Forma de features no esperada: {feats.shape}")

def extract_index(image_path):
    image = Image.open(image_path).convert("RGB")
    x = transform(image).unsqueeze(0)  # [1, 3, 224, 224]
    with torch.no_grad():
        feats = model(x)
        # si algún bloque devuelve una tupla/lista, tomamos el primer tensor
        if isinstance(feats, (list, tuple)):
            feats = feats[0]
        pooled = _global_pool(feats).squeeze(0)  # [C]
        inequality_index = np.std(pooled.cpu().numpy())
    return inequality_index

def compute_inequality(image_5km_path, image_10km_path):
    return {
        "Desigualdad_5km": extract_index(image_5km_path),
        "Desigualdad_10km": extract_index(image_10km_path),
    }

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [125]:
############################################
############## SEGFORMER ###################
############################################

from transformers import SegformerFeatureExtractor, SegformerModel
import torch
from PIL import Image
import numpy as np

# Cargar modelo y habilitar hidden_states
model = SegformerModel.from_pretrained(
    "nvidia/segformer-b0-finetuned-ade-512-512",
    output_hidden_states=True
)
model.eval()

# Procesador (resize, normalize, etc.)
feature_extractor = SegformerFeatureExtractor(do_resize=True, size=224, do_normalize=True)

def _global_pool(feats: torch.Tensor) -> torch.Tensor:
    if feats.dim() == 4:
        return feats.mean(dim=[2,3])   # spatial pooling
    elif feats.dim() == 3:
        return feats.mean(dim=1)       # token pooling
    else:
        raise RuntimeError(f"Forma inesperada: {feats.shape}")

def extract_index(image_path, cut_layer=-4):
    image = Image.open(image_path).convert("RGB")
    inputs = feature_extractor(images=image, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
        hidden_states = outputs.hidden_states  # lista de tensores
        feats = hidden_states[cut_layer]       # elegís la capa (ej: -1 última, -2 penúltima, etc.)
        pooled = _global_pool(feats).squeeze(0)
        return np.std(pooled.cpu().numpy())



def compute_inequality(path_5km, path_10km):
    return {
        "Desigualdad_5km": extract_index(path_5km),
        "Desigualdad_10km": extract_index(path_10km),
    }



/usr/local/lib/python3.12/dist-packages/transformers/models/segformer/feature_extraction_segformer.py:30: FutureWarning: The class SegformerFeatureExtractor is deprecated and will be removed in version 5 of Transformers. Please use SegformerImageProcessor instead.
  warnings.warn(


# OCDE

In [130]:
import pandas as pd
ciudades=pd.read_csv("/content/Gini con latlon OECD.csv", sep=';')

In [100]:
import os
import shutil

# Ruta de origen y destino
src_dir = ocde+"/Imagenes3"
dst_dir = ocde

# Crear lista de archivos
archivos = os.listdir(src_dir)

# Mover sin sobrescribir
for archivo in archivos:
    origen = os.path.join(src_dir, archivo)
    destino = os.path.join(dst_dir, archivo)

    if not os.path.exists(destino):  # si no existe en destino, mover
        shutil.move(origen, destino)
    else:
        print(f"⚠️ Ya existe: {archivo}, no se movió.")

print("✅ Movimiento completado.")

⚠️ Ya existe: Albany - 10K.png, no se movió.
⚠️ Ya existe: Brevard - 5K.png, no se movió.
⚠️ Ya existe: Windsor - 10K.png, no se movió.
⚠️ Ya existe: Dane - 15K.png, no se movió.
⚠️ Ya existe: Gothenburg - 15K.png, no se movió.
⚠️ Ya existe: Saint-Etienne - 5K.png, no se movió.
⚠️ Ya existe: Windsor - 15K.png, no se movió.
⚠️ Ya existe: Rennes - 1K.png, no se movió.
⚠️ Ya existe: Gothenburg - 5K.png, no se movió.
⚠️ Ya existe: Gothenburg - 10K.png, no se movió.
⚠️ Ya existe: Calgary - 1K.png, no se movió.
⚠️ Ya existe: Brevard - 1K.png, no se movió.
⚠️ Ya existe: Dane - 1K.png, no se movió.
⚠️ Ya existe: Dane - 5K.png, no se movió.
⚠️ Ya existe: Brevard - 10K.png, no se movió.
⚠️ Ya existe: Saint-Etienne - 1K.png, no se movió.
⚠️ Ya existe: Nantes - 5K.png, no se movió.
⚠️ Ya existe: Albany - 15K.png, no se movió.
⚠️ Ya existe: Windsor - 5K.png, no se movió.
⚠️ Ya existe: Dane - 10K.png, no se movió.
⚠️ Ya existe: Brevard - 15K.png, no se movió.
⚠️ Ya existe: Gothenburg - 1K.png, no se

In [190]:
ciudades['Desigualdad_5km']=np.nan
ciudades['Desigualdad_10km']=np.nan

from tqdm import tqdm
from PIL import Image
from io import BytesIO

def pil_to_bytes(img):
    buf = BytesIO()
    img.save(buf, format='PNG')
    buf.seek(0)
    return buf

def crop_to_square_center(img, size=1773):
    width, height = img.size
    side = min(width, height, size)
    left = (width - side) // 2
    top = (height - side) // 2
    right = left + side
    bottom = top + side
    return img.crop((left, top, right, bottom))


for i in tqdm(range(len(ciudades)), desc="Procesando ciudades"):
    nombre_aglomerado = ciudades.loc[i, "Ciudad"]

    img_5k_path = os.path.join(ocde, f"{nombre_aglomerado} - 5K.png")
    img_10k_path = os.path.join(ocde, f"{nombre_aglomerado} - 10K.png")


    try:
        # Abrir y recortar imágenes centradas a 1773x1773
        img_5k = crop_to_square_center(Image.open(img_5k_path))
        img_10k = crop_to_square_center(Image.open(img_10k_path))


        # Convertir imágenes a objetos tipo archivo
        img_5k_io = pil_to_bytes(img_5k)
        img_10k_io = pil_to_bytes(img_10k)


        resultados = compute_inequality(img_5k_io,img_10k_io)

        # Guardar en el DataFrame
        ciudades.loc[i, "Desigualdad_5km"] = resultados["Desigualdad_5km"]
        ciudades.loc[i, "Desigualdad_10km"] = resultados["Desigualdad_10km"]

    except Exception as e:
        print(f"⚠️ Error en {nombre_aglomerado}: {e}")

Procesando ciudades: 100%|██████████| 105/105 [04:59<00:00,  2.85s/it]


In [191]:
from scipy.stats import pearsonr

# Eliminar pares con NaN
x = ciudades["Desigualdad_5km"]
y = ciudades["Gini"]
mask = x.notna() & y.notna()

# Calcular correlación de Pearson y p-value
r, p = pearsonr(x[mask], y[mask])

print(f"Coeficiente de Pearson: {r:.3f}")
print(f"Valor p: {p:.5f}")

Coeficiente de Pearson: 0.322
Valor p: 0.00082


# Respuestas Humanas

In [138]:
respuestas=pd.read_csv('respuestas forms.csv', sep=';')

In [193]:
from scipy.stats import pearsonr

# Hacer merge por "Ciudad"
df_merge = ciudades.merge(respuestas[["Ciudad", "Promedio"]], on="Ciudad", how="inner")

# Variables a comparar
x = df_merge["Desigualdad_5km"]
y = df_merge["Promedio"]

# Eliminar NaN
mask = x.notna() & y.notna()

# Calcular correlación
r, p = pearsonr(x[mask], y[mask])

print(f"Coeficiente de Pearson: {r:.3f}")
print(f"Valor p: {p:.5f}")

Coeficiente de Pearson: 0.307
Valor p: 0.11161


# EPH

In [115]:
import pandas as pd
ciudades=pd.read_csv("Gini_EPH.csv")

In [116]:
import numpy as np
ciudades['Desigualdad_5km']=np.nan
ciudades['Desigualdad_10km']=np.nan

In [117]:
import os
import shutil

# Ruta de origen y destino
src_dir = eph+"/Imagenes4"
dst_dir = eph

# Crear lista de archivos
archivos = os.listdir(src_dir)

# Mover sin sobrescribir
for archivo in archivos:
    origen = os.path.join(src_dir, archivo)
    destino = os.path.join(dst_dir, archivo)

    if not os.path.exists(destino):  # si no existe en destino, mover
        shutil.move(origen, destino)
    else:
        print(f"⚠️ Ya existe: {archivo}, no se movió.")

print("✅ Movimiento completado.")

⚠️ Ya existe: Gran La Plata - 5K.png, no se movió.
⚠️ Ya existe: Gran La Plata - 10K.png, no se movió.
⚠️ Ya existe: Gran San Luis - 1K.png, no se movió.
⚠️ Ya existe: Gran La Plata - 1K.png, no se movió.
⚠️ Ya existe: Santiago del Estero-La Banda - 10K.png, no se movió.
⚠️ Ya existe: Gran San Luis - 10K.png, no se movió.
⚠️ Ya existe: Gran San Luis - 5K.png, no se movió.
✅ Movimiento completado.


In [126]:

for i in tqdm(range(len(ciudades)), desc="Procesando ciudades"):
    nombre_aglomerado = ciudades.loc[i, "Nombre_Aglomerado"]

    img_5k_path = os.path.join(eph, f"{nombre_aglomerado} - 5K.png")
    img_10k_path = os.path.join(eph, f"{nombre_aglomerado} - 10K.png")


    try:
        # Abrir y recortar imágenes centradas a 1773x1773
        img_5k = crop_to_square_center(Image.open(img_5k_path))
        img_10k = crop_to_square_center(Image.open(img_10k_path))


        # Convertir imágenes a objetos tipo archivo
        img_5k_io = pil_to_bytes(img_5k)
        img_10k_io = pil_to_bytes(img_10k)


        resultados = compute_inequality(img_5k_io,img_10k_io)

        # Guardar en el DataFrame
        ciudades.loc[i, "Desigualdad_5km"] = resultados["Desigualdad_5km"]
        ciudades.loc[i, "Desigualdad_10km"] = resultados["Desigualdad_10km"]

    except Exception as e:
        print(f"⚠️ Error en {nombre_aglomerado}: {e}")

Procesando ciudades: 100%|██████████| 32/32 [01:28<00:00,  2.77s/it]


In [129]:
from scipy.stats import pearsonr

# Eliminar pares con NaN
x = ciudades["Desigualdad_5km"]
y = ciudades["Gini_Hogares"]
mask = x.notna() & y.notna()

# Calcular correlación de Pearson y p-value
r, p = pearsonr(x[mask], y[mask])

print(f"Coeficiente de Pearson: {r:.3f}")
print(f"Valor p: {p:.5f}")

Coeficiente de Pearson: 0.446
Valor p: 0.01045
